In [237]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.widgets import Button, CheckButtons, TextBox
from scipy.signal import peak_widths
import tkinter as tk
from tkinter import filedialog


%matplotlib qt


In [238]:
# ==========================================
# ZENTRALE KONFIGURATION (Hier anpassen)
# ==========================================
STANDART_DATEI = ()
PLOT_CONFIG = {
    # Fenstergrößen
    "figsize_main": (20, 5),
    "figsize_calib": (13, 8),

    #Subplots
    "subplot_left": 0.075,
    "subplot_bottom": 0.470,
    "subplot_right": 0.970,
    "subplot_top": 0.900,

    #Achsenabschitte
    "xlim": [680, 750], #Wellenlängenbereich
    "ylim": [-0.01, 1.2], #Intensitätsbereich
    
    
    # Layout & Button-Grössen 
    "plot_right_margin": 0.78,  # Rechter Rand des Hauptplots (0 bis 1)
    "btn_x": 0.80,              # X-Position der Buttons & Checkboxen (von links)
    "btn_width": 0.17,          # Breite der Buttons & Checkboxen
    "btn_height": 0.035,        # Höhe der Buttons (kannst du hier verändern!)
    
    # Linien & Marker
    "linewidth_kurve": 0.9,
    "marker_groesse": 6,
    "marker_edge_width": 0.5,
    
    # Schriftgrößen
    "font_size_labels": 15,       
    "font_size_title": 15,        
    "font_size_ticks": 15,         
    "font_size_peaks": 10,         
    "font_size_checkbox": 8,      
    "font_size_button": 8,  

    # Auflösung
     "dpi" : 300  
     
     
     } 


STANDARD_DATEI = r"C:\Users\hanne\Desktop\Prakikum Greifswald\Projekte für Andrei\Plotter\05a2p0kW_10s.txt"
LITERATUR_TOLERANZ_NM = 0.05
FARBPALETTE = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [239]:
def datei_auswaehlen_dialog(initial_dir: Path | str = "") -> str | None:
    root = None
    try:
        root = tk.Tk()
        root.attributes("-topmost", True)
        root.withdraw()
        ausgewaehlter_pfad = filedialog.askopenfilename(
            title="Wähle eine Textdatei aus",
            initialdir=str(initial_dir) if initial_dir else "",
            filetypes=[("Textdateien (*.txt)", "*.txt"), ("Alle Dateien (*.*)", "*.*")]
        )
        return ausgewaehlter_pfad if ausgewaehlter_pfad else None
    except Exception as e:
        print(f"Dateiauswahl übersprungen (Fehler: {e}).")
        return None
    finally:
        if root is not None:
            try: root.destroy()
            except Exception: pass

# Argon-Linien und feste Pixel-Zuordnungen
ARGON_LINIEN_NM: list[tuple[float, str]] = [
    (401.37, "Ar II"), (404.44, "Ar I"), (407.20, "Ar II"), (410.39, "Ar II"),
    (415.86, "Ar I"), (416.41, "Ar I"), (418.19, "Ar I"), (419.10, "Ar I"),
    (422.82, "Ar II"), (425.12, "Ar I"), (425.94, "Ar I"), (426.63, "Ar I"),
    (427.22, "Ar I"), (430.01, "Ar I"), (437.13, "Ar II"), (437.97, "Ar II"),
    (440.01, "Ar II"), (442.60, "Ar II"), (451.07, "Ar I"), (454.51, "Ar II"),
    (457.93, "Ar II"), (458.99, "Ar II"), (460.96, "Ar II"), (465.79, "Ar II"),
    (470.23, "Ar I"), (472.68, "Ar II"), (473.59, "Ar II"), (480.60, "Ar II"),
    (484.78, "Ar II"), (487.99, "Ar II"), (493.32, "Ar II"), (496.51, "Ar II"),
    (500.94, "Ar II"), (506.20, "Ar II"), (514.53, "Ar II"), (516.23, "Ar I"),
    (518.78, "Ar I"), (545.17, "Ar I"), (549.59, "Ar I"), (555.87, "Ar I"),
    (560.67, "Ar I"), (565.07, "Ar I"), (573.95, "Ar I"), (588.86, "Ar I"),
    (591.21, "Ar I"), (603.21, "Ar I"), (604.32, "Ar I"), (605.94, "Ar I"),
    (617.31, "Ar I"), (621.59, "Ar I"), (629.69, "Ar I"), (636.96, "Ar I"),
    (638.47, "Ar I"), (664.37, "Ar II"), (667.73, "Ar I"), (675.28, "Ar I"),
    (687.13, "Ar I"), (693.77, "Ar I"), (696.54, "Ar I"), (703.03, "Ar I"),
    (706.72, "Ar I"), (714.71, "Ar I"), (720.70, "Ar I"), (727.29, "Ar I"),
    (735.33, "Ar I"), (737.21, "Ar I"), (738.40, "Ar I")
]

FESTE_PIXEL_PEAKS: list[tuple[int, float]] = [
    (733,  401.37), (760,  404.44), (785,  407.20), (813,  410.39), (862,  415.86),
    (867,  416.41), (883,  418.19), (891,  419.10), (924,  422.82), (945,  425.12),
    (952,  425.94), (958,  426.63), (963,  427.22), (988,  430.01), (1050, 437.13),
    (1057, 437.97), (1076, 440.01), (1098, 442.60), (1173, 451.07), (1203, 454.51),
    (1233, 457.93), (1243, 458.99), (1259, 460.96), (1301, 465.79), (1340, 470.23),
    (1361, 472.68), (1369, 473.59), (1431, 480.60), (1467, 484.78), (1495, 487.99),
    (1540, 493.32), (1568, 496.51), (1606, 500.94), (1652, 506.20), (1724, 514.53),
    (1738, 516.23), (1760, 518.78), (1986, 545.17), (2023, 549.59), (2077, 555.87),
    (2118, 560.67), (2156, 565.07), (2231, 573.95), (2357, 588.86), (2377, 591.21),
    (2479, 603.21), (2488, 604.32), (2502, 605.94), (2597, 617.31), (2633, 621.59),
    (2702, 629.69), (2763, 636.96), (2776, 638.47), (2993, 664.37), (3022, 667.73),
    (3084, 675.28), (3184, 687.13), (3239, 693.77), (3263, 696.54), (3316, 703.03),
    (3347, 706.72), (3414, 714.71), (3464, 720.70), (3520, 727.29), (3586, 735.33),
    (3602, 737.21), (3612, 738.40)
]

def finde_literaturwert(wellenlaenge: float) -> tuple[float, str] | None:
    abstaende = [abs(wellenlaenge - lit_wert) for lit_wert, _ in ARGON_LINIEN_NM]
    index = int(np.argmin(abstaende))
    if abstaende[index] > LITERATUR_TOLERANZ_NM:
        return None
    return ARGON_LINIEN_NM[index]

def lade_spektren(pfad: Path) -> dict[str, dict[str, np.ndarray]]:
    rohdaten = np.loadtxt(pfad, delimiter="\t", skiprows=1)
    if rohdaten.ndim == 1: rohdaten = rohdaten.reshape(1, -1)
    anzahl_spalten = rohdaten.shape[1]
    serien = {}
    for i in range(anzahl_spalten // 2):
        name = pfad.stem if anzahl_spalten // 2 == 1 else f"{pfad.stem} (Spalte {i + 1})"
        serien[name] = {
            "wellenlaenge": rohdaten[:, 2 * i],
            "intensitaet": rohdaten[:, 2 * i + 1],
            "pixel": np.arange(rohdaten.shape[0], dtype=float),
        }
    return serien

In [240]:
class SpektrumViewer:
    def __init__(self, serien: dict[str, dict[str, np.ndarray]], titel: str):
        self.serien = serien
        self.farben = {name: FARBPALETTE[i % len(FARBPALETTE)] for i, name in enumerate(serien)}
        self.sichtbar = {name: True for name in serien}
        self.sichtbar["Peaks"] = True

        self.linien, self.peak_grafik = {}, {}
        self.ausgewaehlte_peaks = {name: [] for name in serien}
        self.label_map, self.kalibrier_fig = {}, None
        
        # Platzhalter für Widgets zur Vermeidung von Garbage Collection
        self.btn_reset = None
        self.btn_auto = None
        self.btn_kalib = None
        self.btn_add = None
        self.check = None
        self.text_box_grad = None
        self.btn_kalib_ok = None

        self._erstelle_fenster(titel)
        self._zeichne_kurven()
        self._erstelle_steuerelemente()
        self._skalierung_aktualisieren()
        self._aktiviere_klick_auswahl()
        self._aktiviere_koordinaten_anzeige()

    def _erstelle_fenster(self, titel: str) -> None:
        self.fig, self.ax = plt.subplots(figsize=PLOT_CONFIG["figsize_main"])
        #dpi = PLOT_CONFIG["figsize_main"]
        self.ax.minorticks_on()
        
        if self.fig.canvas.manager is not None:
            self.fig.canvas.manager.set_window_title(f"Spektrum-Viewer - {titel}")
            try: self.fig.canvas.manager.window.state("zoomed")
            except Exception: pass

        self.fig.subplots_adjust(
            left=PLOT_CONFIG.get("subplot_left", 0.1),
            bottom=PLOT_CONFIG.get("subplot_bottom", 0.12),
            right=PLOT_CONFIG["subplot_right"],
            top=PLOT_CONFIG.get("subplot_top", 0.9)
        )
            
        # Rechter Rand des Hauptdiagramms aus Config
        # self.fig.subplots_adjust(right=PLOT_CONFIG["plot_right_margin"], bottom=0.12)

        # Button-Werte aus der Config auslesen
        bx = PLOT_CONFIG["btn_x"]
        bw = PLOT_CONFIG["btn_width"]
        bh = PLOT_CONFIG["btn_height"]
        abstand = 0.007  # Kleiner vertikaler Zwischenraum

        # Dynamisches Stapeln von unten nach oben
        y_reset = 0.03
        y_auto = y_reset + bh + abstand
        y_kalib = y_auto + bh + abstand
        y_add = y_kalib + bh + abstand

        # Buttons erstellen
        self.btn_reset = Button(self.fig.add_axes([bx, y_reset, bw, bh]), "Peaks zurücksetzen")
        self.btn_reset.label.set_fontsize(PLOT_CONFIG["font_size_button"])
        self.btn_reset.on_clicked(self._peaks_zuruecksetzen)

        self.btn_auto = Button(self.fig.add_axes([bx, y_auto, bw, bh]), "Peaks auswählen")
        self.btn_auto.label.set_fontsize(PLOT_CONFIG["font_size_button"])
        self.btn_auto.on_clicked(self._peaks_automatisch_finden)

        self.btn_kalib = Button(self.fig.add_axes([bx, y_kalib, bw, bh]), "Peaks bestätigen")
        self.btn_kalib.label.set_fontsize(PLOT_CONFIG["font_size_button"])
        self.btn_kalib.on_clicked(self._kalibrierung_starten)

        self.btn_add = Button(self.fig.add_axes([bx, y_add, bw, bh]), "Datei hinzufügen")
        self.btn_add.label.set_fontsize(PLOT_CONFIG["font_size_button"])
        self.btn_add.on_clicked(self._datei_hinzuufuegen)

        # Beschriftungen des Hauptplots
        self.ax.set_xlabel("wavelength [nm]", fontsize=PLOT_CONFIG["font_size_labels"])
        self.ax.set_ylabel("intensity [counts/s]", fontsize=PLOT_CONFIG["font_size_labels"])
        self.ax.set_title("Wellenlängenspektrum", fontsize=PLOT_CONFIG["font_size_title"])
        self.ax.tick_params(axis='both', labelsize=PLOT_CONFIG["font_size_ticks"])
        self.ax.grid(True, alpha=0.3)

    def _erstelle_steuerelemente(self) -> None:
        if hasattr(self, "ax_check") and self.ax_check in self.fig.axes:
            self.ax_check.remove()
        self.label_map.clear()
        labels_kurz, serien_farben = [], []
        for name in self.serien.keys():
            kurz = (name[:16] + "...") if len(name) > 19 else name
            basis_kurz, zaehler = kurz, 1
            while kurz in self.label_map:
                kurz = f"{basis_kurz[:13]}..({zaehler})"
                zaehler += 1
            self.label_map[kurz] = name
            labels_kurz.append(kurz)
            serien_farben.append(self.farben[name])

        aktive = [self.sichtbar.get(name, True) for name in self.serien.keys()]
        
        # Button-Werte aus der Config auslesen
        bh = PLOT_CONFIG["btn_height"]
        abstand = 0.007
        y_reset = 0.03
        y_auto = y_reset + bh + abstand
        y_kalib = y_auto + bh + abstand
        y_add = y_kalib + bh + abstand
        
        # Checkboxen direkt über dem "Datei hinzufügen"-Button platzieren
        y_check_start = y_add + bh + 0.012
        
        # Dynamische Höhe begrenzen, damit sie nicht aus dem Bild ragt
        height = min(0.52, max(0.10, 0.032 * len(labels_kurz) + 0.02))
        
        # Achse sauber und nur einmalig erstellen
        self.ax_check = self.fig.add_axes([PLOT_CONFIG["btn_x"], y_check_start, PLOT_CONFIG["btn_width"], height])
        
        self.check = CheckButtons(ax=self.ax_check, labels=labels_kurz, actives=aktive)
        for text_label, farbe in zip(self.check.labels, serien_farben):
            text_label.set_color(farbe)
            text_label.set_fontsize(PLOT_CONFIG["font_size_checkbox"])

        self.check.on_clicked(lambda l: (self.sichtbar.update({self.label_map[l]: not self.sichtbar.get(self.label_map[l], True)}), self._wende_sichtbarkeit_an(), self._skalierung_aktualisieren(), self.fig.canvas.draw_idle()))
    def _zeichne_kurven(self) -> None:
        for name, daten in self.serien.items():
            (linie,) = self.ax.plot(
                daten["wellenlaenge"], daten["intensitaet"],
                label=name, color=self.farben[name], linewidth=PLOT_CONFIG["linewidth_kurve"],
            )
            self.linien[name] = linie

    def _skalierung_aktualisieren(self) -> None:
        sichtbare = [n for n, s in self.sichtbar.items() if s and n in self.serien]
        if not sichtbare: return
        
        # --- Y-Achse ---
        if PLOT_CONFIG.get("ylim") is not None:
            self.ax.set_ylim(PLOT_CONFIG["ylim"])
        else:
            ymin = min(np.min(self.serien[n]["intensitaet"]) for n in sichtbare)
            ymax = max(np.max(self.serien[n]["intensitaet"]) for n in sichtbare)
            y_pruf = (ymax - ymin) * 0.05 or 1.0
            self.ax.set_ylim(ymin - y_pruf, ymax + y_pruf)
            
        # --- X-Achse ---
        if PLOT_CONFIG.get("xlim") is not None:
            self.ax.set_xlim(PLOT_CONFIG["xlim"])
        else:
            xmin = min(np.min(self.serien[n]["wellenlaenge"]) for n in sichtbare)
            xmax = max(np.max(self.serien[n]["wellenlaenge"]) for n in sichtbare)
            x_pruf = (xmax - xmin) * 0.02 or 1.0
            self.ax.set_xlim(xmin - x_pruf, xmax + x_pruf)
   
    def _datei_hinzuufuegen(self, ereignis=None) -> None:
        pfad_str = datei_auswaehlen_dialog(Path(STANDARD_DATEI).parent)
        if not pfad_str or not Path(pfad_str).exists(): return
        neue_serien = lade_spektren(Path(pfad_str))
        vorher = len(self.serien)
        for i, (name, daten) in enumerate(neue_serien.items()):
            basis, z = name, 1
            while name in self.serien:
                name = f"{basis} ({z})"
                z += 1
            self.serien[name], self.farben[name], self.sichtbar[name], self.ausgewaehlte_peaks[name] = daten, FARBPALETTE[(vorher + i) % len(FARBPALETTE)], True, []
            (l,) = self.ax.plot(daten["wellenlaenge"], daten["intensitaet"], label=name, color=self.farben[name], linewidth=PLOT_CONFIG["linewidth_kurve"])
            self.linien[name] = l
        self._erstelle_steuerelemente()
        self._skalierung_aktualisieren()
        self.fig.canvas.draw_idle()

    def _entferne_alte_peak_grafik(self) -> None:
        for elemente in self.peak_grafik.values():
            for el in elemente["marker"] + elemente["beschriftungen"]:
                try: el.remove()
                except Exception: pass

    def _zeichne_peaks_einer_serie(self, name: str, peaks: list[dict], farbe: str) -> dict:
        marker, beschriftungen = [], []
        for p in peaks:
            x = p.get("mess_wellenlaenge", p["wellenlaenge"])
            f_col = "red" if p.get("spezies") == "Ar I" else ("blue" if p.get("spezies") == "Ar II" else farbe)
            (m,) = self.ax.plot(x, p["intensitaet"], "^", color=f_col, markersize=PLOT_CONFIG["marker_groesse"], markeredgecolor="black", markeredgewidth=PLOT_CONFIG["marker_edge_width"])
            txt_str = f"{p['spezies']} {p['lit_wert']:.2f}" if "spezies" in p and "lit_wert" in p else f"Pixel {int(p['pixel'])} | {x:.2f} nm"
            t = self.ax.annotate(txt_str, xy=(x, p["intensitaet"]), xytext=(0, 10), textcoords="offset points", rotation=90, fontsize=PLOT_CONFIG["font_size_peaks"], ha="center", va="bottom", color=f_col)
            marker.append(m); beschriftungen.append(t)
        return {"marker": marker, "beschriftungen": beschriftungen}

    def _peaks_zuruecksetzen(self, ereignis=None) -> None:
        for n in self.ausgewaehlte_peaks: self.ausgewaehlte_peaks[n].clear()
        self._aktualisiere_peak_anzeige()

    def _peaks_automatisch_finden(self, ereignis=None) -> None:
        for name, daten in self.serien.items():
            if not self.sichtbar.get(name, True):
                self.ausgewaehlte_peaks[name] = []
                continue
            self.ausgewaehlte_peaks[name] = []
            for pix, lit_wert in FESTE_PIXEL_PEAKS:
                if pix > len(daten["pixel"]) - 1: continue
                idx = int(pix)
                wl_akt = daten["wellenlaenge"][idx]
                treffer = finde_literaturwert(lit_wert)
                spezies = treffer[1] if treffer else "Unbekannt"
                try:
                    _, h, l_idx, r_idx = peak_widths(daten["intensitaet"], [idx], rel_height=0.5)
                    l_wl = np.interp(l_idx[0], np.arange(len(daten["pixel"])), daten["wellenlaenge"])
                    r_wl = np.interp(r_idx[0], np.arange(len(daten["pixel"])), daten["wellenlaenge"])
                    fwhm, halbmax = r_wl - l_wl, h[0]
                except Exception:
                    fwhm, halbmax, l_wl, r_wl = 0.0, daten["intensitaet"][idx] / 2, wl_akt, wl_akt

                self.ausgewaehlte_peaks[name].append({
                    "pixel": float(idx), "wellenlaenge": lit_wert, "mess_wellenlaenge": wl_akt,
                    "intensitaet": daten["intensitaet"][idx], "fwhm": fwhm, "halbmax_hoehe": halbmax,
                    "links_wl": l_wl, "rechts_wl": r_wl, "spezies": spezies, "lit_wert": lit_wert,
                })
        self._aktualisiere_peak_anzeige()

    def _wende_sichtbarkeit_an(self) -> None:
        for name, linie in self.linien.items(): linie.set_visible(self.sichtbar.get(name, True))
        p_glob = self.sichtbar.get("Peaks", True)
        for name, el in self.peak_grafik.items():
            s = self.sichtbar.get(name, True) and p_glob
            for m in el["marker"]: m.set_visible(s)
            for t in el["beschriftungen"]: t.set_visible(s)

    def _aktiviere_klick_auswahl(self) -> None:
        def bei_klick(e) -> None:
            if e.inaxes != self.ax or e.xdata is None or e.button != 3: return
            sichtbare = [n for n, s in self.sichtbar.items() if s and n in self.serien]
            if not sichtbare: return
            daten = self.serien[sichtbare[0]]
            k_idx = (np.abs(daten["wellenlaenge"] - e.xdata)).argmin()
            start, ende = max(0, k_idx - 3), min(len(daten["wellenlaenge"]), k_idx + 3)
            if ende - start <= 0: return
            idx = start + np.argmax(daten["intensitaet"][start:ende])
            _, h, l_idx, r_idx = peak_widths(daten["intensitaet"], [idx], rel_height=0.5)
            l_wl = np.interp(l_idx[0], np.arange(len(daten["pixel"])), daten["wellenlaenge"])
            r_wl = np.interp(r_idx[0], np.arange(len(daten["pixel"])), daten["wellenlaenge"])
            wl_akt = daten["wellenlaenge"][idx]
            p_d = {"pixel": float(idx), "wellenlaenge": wl_akt, "mess_wellenlaenge": wl_akt, "intensitaet": daten["intensitaet"][idx], "fwhm": r_wl - l_wl, "halbmax_hoehe": h[0], "links_wl": l_wl, "rechts_wl": r_wl}
            treffer = finde_literaturwert(wl_akt)
            if treffer: p_d["spezies"], p_d["lit_wert"] = treffer[1], treffer[0]
            
            existiert = False
            for p in self.ausgewaehlte_peaks[sichtbare[0]]:
                if abs(p["pixel"] - p_d["pixel"]) < 2:
                    self.ausgewaehlte_peaks[sichtbare[0]].remove(p)
                    existiert = True
                    break
            if not existiert: self.ausgewaehlte_peaks[sichtbare[0]].append(p_d)
            self._aktualisiere_peak_anzeige()
        self.fig.canvas.mpl_connect("button_press_event", bei_klick)

    def _aktualisiere_peak_anzeige(self) -> None:
        self._entferne_alte_peak_grafik()
        self.peak_grafik = {n: self._zeichne_peaks_einer_serie(n, p, self.farben[n]) for n, p in self.ausgewaehlte_peaks.items()}
        self._wende_sichtbarkeit_an()
        self.fig.canvas.draw_idle()

    def _kalibrierung_starten(self, ereignis=None) -> None:
        pixel, wellenlaenge_lit, serienname = [], [], []
        for name, peaks in self.ausgewaehlte_peaks.items():
            for p in peaks:
                if "lit_wert" not in p:
                    treffer = finde_literaturwert(p["wellenlaenge"])
                    if not treffer: continue
                    p["lit_wert"], p["spezies"] = treffer[0], treffer[1]
                pixel.append(float(p["pixel"]))
                wellenlaenge_lit.append(p["lit_wert"])
                serienname.append(name)
        if len(pixel) < 3: return
        pixel, wellenlaenge_lit, serienname = np.asarray(pixel), np.asarray(wellenlaenge_lit), np.asarray(serienname)
        unique_pix, unique_idx = np.unique(pixel, return_index=True)
        pixel, wellenlaenge_lit, serienname = unique_pix, wellenlaenge_lit[unique_idx], serienname[unique_idx]
        
        self.kalibrier_fig = plt.figure(figsize=PLOT_CONFIG["figsize_calib"])
        ax_fit = self.kalibrier_fig.add_axes([0.10, 0.42, 0.65, 0.48])
        ax_res = self.kalibrier_fig.add_axes([0.10, 0.15, 0.65, 0.20])
        
        self.kalibrier_fig.text(0.80, 0.74, "Polynom-Grad eingeben:", fontsize=9, va="bottom")
        self.text_box_grad = TextBox(self.kalibrier_fig.add_axes([0.80, 0.66, 0.15, 0.05]), "", initial="3")
        
        self.btn_kalib_ok = Button(self.kalibrier_fig.add_axes([0.80, 0.45, 0.15, 0.08]), "Peaks bestätigen")
        self.btn_kalib_ok.on_clicked(lambda e: plt.close(self.kalibrier_fig))

        def aktualisiere_plot(text_grad):
            try: grad = int(text_grad)
            except ValueError: return
            ax_fit.clear(); ax_res.clear()
            if len(pixel) <= grad: return
            koeff = np.polyfit(pixel, wellenlaenge_lit, grad)
            p_poly = np.poly1d(koeff)
            x_fit = np.linspace(pixel.min(), pixel.max(), 500)
            res = wellenlaenge_lit - p_poly(pixel)
            for name in self.serien:
                mask = serienname == name
                if np.any(mask):
                    ax_fit.plot(pixel[mask], wellenlaenge_lit[mask], "o", markersize=5, color=self.farben[name], label=name)
                    ax_res.plot(pixel[mask], res[mask], "o", markersize=5, color=self.farben[name])
            ax_fit.plot(x_fit, p_poly(x_fit), "-", color="red", linewidth=1.5, label="Fit-Kurve")
            ax_fit.set_ylabel("Wellenlänge [nm]"); ax_fit.grid(True, alpha=0.3); ax_fit.legend(fontsize=8)
            ax_res.axhline(0, linestyle="--", color="black"); ax_res.set_xlabel("Pixel"); ax_res.set_ylabel("Residuem [nm]"); ax_res.grid(True, alpha=0.3)
            self.kalibrier_fig.canvas.draw_idle()

        self.text_box_grad.on_submit(aktualisiere_plot)
        aktualisiere_plot("3")
        plt.show()

    def _aktiviere_koordinaten_anzeige(self) -> None:
        def format_coord(x, y):
            items = list(self.serien.items())
            if not items: return f"x={x:.2f}, y={y:.2f}"
            teile = []
            for name, daten in items:
                if not self.sichtbar.get(name, True): continue
                idx = np.clip(np.searchsorted(daten["wellenlaenge"], x), 1, len(daten["wellenlaenge"]) - 1)
                if abs(daten["wellenlaenge"][idx - 1] - x) < abs(daten["wellenlaenge"][idx] - x): idx -= 1
                teile.append(f"{name}: {daten['intensitaet'][idx]:.1f} a.u. ({daten['wellenlaenge'][idx]:.2f} nm)")
            return f"Wellenlänge: {x:.2f} nm | " + " | ".join(teile)
        self.ax.format_coord = format_coord

In [241]:
pfad_str = datei_auswaehlen_dialog(Path(STANDARD_DATEI).parent)
if pfad_str: STANDARD_DATEI = pfad_str

pfad = Path(STANDARD_DATEI)
if not pfad.exists():
    print(f"Fehler: Datei '{pfad}' nicht gefunden.")
else:
    serien = lade_spektren(pfad)
    viewer = SpektrumViewer(serien, titel=pfad.name)
    plt.show()